## Hybrid Search:
---

Retrieval Augmented Generation, also known as RAG, is a powerful method to find relevant documents in a corpus of information, which you then provide to an LLM to give answers to user questions.

Traditionally, RAG first uses vector similarity to find relevant chunks of documents in the corpus and then feeds the most relevant chunks into the LLM to provide a response.

This works really well in a lot of scenarios since semantic similarity is a powerful way to find the most relevant chunks. However, semantic similarity struggles in some scenarios, for example, when a user inputs specific keywords or IDs that need to be explicitly located to be used as a relevant chunk. In these instances, vector similarity is not that effective, and you need a better approach to find the most relevant chunks.

This is where keyword search comes in, where you find relevant chunks while using keyword search and vector similarity, also known as hybrid search.

- However, vector similarity search is not as good at precise matching on keywords, abbreviations, and names, which can get lost in vector embeddings along with the surrounding words. Here, keyword search performs better.

* Use Cases for Hybrid Search:
   - Matching abbreviations like GAN or LLaMA in the query and knowledge base.
   - Identifying exact words of names or objects, like “Biden” or “Salvador Dali".
   - Identifying exact code snippets in programming languages. (Taking a similarity search approach in such instances is practically useless.)

* Keyword search guarantees that abbreviations, annotations, names, and code stay on the radar, while vector search ensures that search results are contextually relevant. 

* Some limitations:
  - `Latency`: Hybrid search involves performing two search algorithms, so it may be slower than a semantic search when executing on a large knowledge corpus.
  - `Computational Expense`: Developing and customizing models for hybrid search can be computationally expensive. It's best to consider hybrid search only if your system requires keyword-backed results.
 - `Native Support in Databases`: Not all vector databases support hybrid search. You need to ensure the vector database you choose does

* Keyword Search:
Sparse vectors are vectors with high dimensionality, where most elements are zero. They usually symbolize various language tokens, non-zero values signifying their respective importance. Keyword search is also called sparse vector search. The BM25 (Best Match 25) algorithm is a popular and effective ranking function employed for keyword matching in embeddings. BM25 finds the most relevant documents for a given query by examining two things:
 1. How often do the query words appear in each document? (the more, the better)
 2. How rare are the query words across all the documents? (the rarer, the better)

* Vector Search:
Dense vectors, or embeddings are arrays with a high number of dimensions, filled predominantly with meaningful, non-zero values. Machine learning frequently employs these to represent the underlying semantics and connections of words in a numerical format, thereby effectively encapsulating their semantic essence. Dense vector search is a method used in semantic search systems for finding similar items in a vector space.
A common approach to vector search is cosine similarity search.

* Combination of these 2 search:
   H=(1−α)K+αV
   Where:
     - H is the hybrid search score
     - α is the weighted parameter
     - K is the keyword search score
     - V is the vector search score

**Reciprocal Rank Fusion (RRF)** is one of several available methods for combining dense and sparse search scores. RRF ranks each passage according to its place in the keyword and vector outcome lists, and then merges these rankings to generate a unified result list. The RRF score is determined by summing the inverse rankings from each list. Positioning the document’s rank in the denominator imposes a penalty on documents that appear lower in the list.

In [1]:
from langchain_community.retrievers import BM25Retriever
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [2]:
hf_api_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [3]:
loader = PyPDFLoader("Data/rag_paper.pdf")
docs = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 30
)
chunks = splitter.split_documents(docs)

In [5]:
len(chunks)

419

In [6]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
embedding = HuggingFaceEndpointEmbeddings(
    model = "BAAI/bge-base-en-v1.5",
    huggingfacehub_api_token = hf_api_key
)

In [7]:
from langchain_community.vectorstores import Chroma
vector_store = Chroma.from_documents(
    documents = chunks,
    embedding = embedding
)

In [8]:
vectorstore_retriever = vector_store.as_retriever(
    search_kwargs = {"k" : 3}
)

In [9]:
keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever.k = 3

In [10]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(
    retrievers = [vectorstore_retriever , keyword_retriever] , weights = [0.5 , 0.5]
)

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline 
from langchain_huggingface import HuggingFacePipeline

In [12]:
# function for loading 4-bit quantized model
def load_quantized_model(model_name: str):
    """
    model_name: Name or path of the model to be loaded.
    return: Loaded quantized model.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        quantization_config=bnb_config,
    )
    return model

In [13]:
# initializing tokenizer
def initialize_tokenizer(model_name: str):
    """
    model_name: Name or path of the model for tokenizer initialization.
    return: Initialized tokenizer.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, return_token_type_ids=False)
    tokenizer.bos_token_id = 1  # Set beginning of sentence token id
    return tokenizer

In [14]:
model_name = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = initialize_tokenizer(model_name)
model = load_quantized_model(model_name)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [15]:
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    use_cache=True,
    device_map="auto",
    max_length=2048,
    do_sample=True,
    top_k=5,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences', 'pad_token_id', 'use_cache', 'top_k', 'do_sample', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
